# X API Agent

The X API Agent is an AI-powered social media assistant that interacts with X (Twitter) via the official API. It can post tweets, look up user profiles, and search recent posts — all through natural language instructions, using OAuth 1.0a authentication.

In [1]:
!pip install -q aixplain tweepy

## Prerequisites

### aiXplain API key

Get your team access key from [platform.aixplain.com → Integrations](https://platform.aixplain.com/account/integrations).

### Connect X in Studio (required before using the tool)

The agent uses the **X (Twitter) integration** from aiXplain Studio. You must **connect that integration in the Studio UI first**; only then can you load it as a tool in code.

1. Open [studio.aixplain.com](https://studio.aixplain.com/).
2. Go to **Integrations** and add or open **X (Twitter)**.
3. Complete the connection flow (you need a project and app from the [X Developer Portal](https://developer.x.com/en/portal/dashboard) with **Read and Write** permissions, and the **API Key**, **API Secret**, **Access Token**, and **Access Token Secret** from the **Keys and Tokens** tab when Studio prompts you).
4. After X shows as connected, open **Tools** in Studio and copy your X tool’s path or ID for the `Tool.get(...)` cell below.

In [ ]:
AixplainAPIKey  = "<YOUR_AIXPLAIN_API_KEY>"

import os
os.environ["TEAM_API_KEY"]    = AixplainAPIKey


In [3]:
import inspect
from aixplain import Aixplain

aix = Aixplain(AixplainAPIKey)

# What is the X API Agent?

The goal of this agent is to make X (Twitter) interactions programmable and conversational. You can instruct it to compose and post tweets, look up accounts, or search for recent posts — all without manually calling the tool.

## Agent Architecture

After you connect **X (Twitter)** in Studio, you attach that integration as **one tool** on the agent. It exposes X API v2 capabilities (for example via Composio), including:

- **Post Tweet** — posts a tweet on behalf of the authenticated user.
- **Get User Info** — retrieves a user's public profile: follower count, following count, bio, and recent tweet count.
- **Search Recent Tweets** — searches tweets from the last 7 days matching a keyword or phrase.

## Agent Workflow

1. **Understand the request** — determine whether the user wants to post, look up a profile, or search.
2. **Call the right tool** — route to post, profile lookup, or search accordingly.
3. **Report back** — return the tweet URL, profile data, or search results.

## Load the X tool

Use the identifier from Studio for the integration you connected above. Replace the string in the next cell with your tool path (for example `your-team/your-x-tool/composio`).

If the integration is not connected in Studio, this step will fail or the agent will not be able to call X.

In [ ]:
x_tool = aix.Tool.get("asma-furniturewala/twitter-c3ad/composio")


# Build Your Agent

To create an agent, define:

* A unique **name** and **description** for its purpose.
* The **tools** it will use — here, the single **X integration tool** you connected in Studio (it covers posting, profile lookup, and tweet search).
* The **instructions** that guide the agent's behaviour and safety constraints.

In [33]:
x_agent = aix.Agent(
    name="X Agent",
    description="Social media agent that posts tweets, looks up X user profiles, and searches recent posts using the official X API v2.",
    instructions="""
    You are a social media assistant with access to the X (Twitter) API via the connected integration.
    Use the available X actions as follows:

    - Post Tweet: posts a tweet on behalf of the authenticated user. Use this when
      the user explicitly asks you to post, tweet, or publish something.
      Always show the user the exact tweet text before posting and confirm.
      Never post unless explicitly instructed.

    - Get X User Info: looks up a public profile by username. Use this when the
      user asks about an account, follower count, or bio.

    - Search Recent Tweets: searches tweets from the last 7 days. Use this for
      trending topics, sentiment checks, or finding what people are saying about
      a keyword, hashtag, or account.

    Guidelines:
    - Tweets must be 280 characters or fewer — count carefully before posting.
    - When composing tweet text, make it concise, engaging, and on-brand.
    - For search queries, use X search operators where helpful
      (e.g. 'lang:en -is:retweet', '#hashtag', 'from:username').
    - Always return the tweet URL after a successful post.
    - Never include harmful, misleading, or policy-violating content in tweets.
    """,
    tools=[x_tool],
)
x_agent.save()
print(f"Agent created: {x_agent.id}")

Agent created: 69c539859e1dff6231a1b3db


# Run your Agent

To ensure your agent is performing as expected, run it using the `run` method with sample inputs. Analyze the outputs, verify their accuracy, and debug any issues by inspecting the agent's configurations, tools, and instructions.

In [ ]:
# Look up a user profile
result = x_agent.run("Look up the X profile for @OpenAI")
print(result.data.output)

In [ ]:
# Search recent tweets
result2 = x_agent.run("What are people saying about AI agents on X this week?")
print(result2.data.output)

In [ ]:
# Compose and post a tweet
result3 = x_agent.run("Post a tweet announcing that I just built an AI agent using aiXplain")
print(result3.data.output)

In [ ]:
# Multi-turn: refine the tweet before posting
result4 = x_agent.run(
    "Make it shorter and add the hashtag #aiXplain",
    session_id=result3.data.session_id
)
print(result4.data.output)

# Save the Agent

Once you are happy with your agent, save it to access the agent endpoints.

In [ ]:
x_agent.save()

aiXplain empowers you to seamlessly build, customize, and deploy intelligent agents tailored to your specific needs. Whether you're creating standalone agents or advanced multi-agent systems, the platform provides a robust toolkit for integrating cutting-edge AI capabilities into your workflows. To learn more, visit [aiXplain](https://aixplain.com).